# 8. Protein model validation partition plots

This notebook reads the saved external validation predictions/metrics produced by `7. test_8020.ipynb` and creates final model-partition plots for each validation dataset.

For every validation dataset and every metric, proteins are assigned to the model/run that predicts them best:

- Pearson: highest `test_pearson_corr`
- Cosine: highest `test_cosine_similarity`
- MSE: lowest `test_mse`

Proteins that do not pass the quality threshold are placed into a red bad segment. Small good model segments are grouped into `Other good models` to keep the plots clean.


In [ ]:
from pathlib import Path
import sys

# Make the adjacent src/ package importable when this notebook is run from the notebooks folder.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import src.utils_val as uv

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


In [ ]:
OUTPUT_DIR = Path("../results_comparison_all_models")
METRICS_PROTEIN_CSV = OUTPUT_DIR / "eval_metrics_by_protein.csv"
METRICS_TISSUE_CSV = OUTPUT_DIR / "eval_metrics_by_tissue.csv"
PREDICTIONS_PARQUET = OUTPUT_DIR / "eval_predictions_long.parquet"
PREDICTIONS_CSV = OUTPUT_DIR / "eval_predictions_long.csv"

PEARSON_GOOD_THRESHOLD = 0.30
COSINE_GOOD_THRESHOLD = 0.30
MSE_GOOD_PERCENTILE = 25  # lower MSE is better; threshold is this percentile within each dataset
MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT = 200

if not METRICS_PROTEIN_CSV.exists():
    raise FileNotFoundError(f"Missing {METRICS_PROTEIN_CSV}. Run 7. test_8020.ipynb first.")

metrics_by_protein = pd.read_csv(METRICS_PROTEIN_CSV)
metrics_by_tissue = pd.read_csv(METRICS_TISSUE_CSV) if METRICS_TISSUE_CSV.exists() else pd.DataFrame()

print("Loaded protein metrics:", metrics_by_protein.shape)
print("Datasets:", sorted(metrics_by_protein["dataset"].dropna().unique()))
print("Run labels:", metrics_by_protein["run_label"].nunique())

display(metrics_by_protein.head())


## 1. Concise overview

In [ ]:
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Skip exact k50 and k500 models from the whole analysis
skip_k_pattern = r"\bk(?:50|500)\b"

metrics_filtered = metrics_by_protein[
    ~metrics_by_protein["run_label"]
    .astype(str)
    .str.contains(skip_k_pattern, case=False, regex=True, na=False)
].copy()

overview = (
    metrics_filtered.groupby(["dataset", "run_label"])
    .agg(
        proteins=("protein", "nunique"),
        median_pearson=("test_pearson_corr", "median"),
        median_cosine=("test_cosine_similarity", "median"),
        median_mse=("test_mse", "median"),
    )
    .reset_index()
)

print("Model/run coverage by dataset, excluding k50 and k500")
display(
    overview.sort_values(
        ["dataset", "median_pearson"],
        ascending=[True, False]
    )
)

# Cleaner run labels
overview["run_short"] = (
    overview["run_label"]
    .astype(str)
    .str.replace(" | ", "\n", regex=False)
)

# Consistent run order: best median Pearson overall first
run_order = (
    overview.groupby("run_short")["median_pearson"]
    .median()
    .sort_values(ascending=False)
    .index
)

metric_specs = [
    ("median_pearson", "Median Pearson", False),
    ("median_cosine", "Median cosine", False),
    ("median_mse", "Median MSE", True),
]

sns.set_theme(style="whitegrid", context="notebook")

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
    constrained_layout=True,
    sharex=True
)

for ax, (metric, title, use_log) in zip(axes, metric_specs):
    sns.barplot(
        data=overview,
        x="dataset",
        y=metric,
        hue="run_short",
        hue_order=run_order,
        ax=ax,
        errorbar=None,
        width=0.75
    )

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(title)
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="x", visible=False)

    if use_log:
        ax.set_yscale("log")

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# Put one clean legend outside the panels
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title="Run",
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
    ncol=1
)

sns.despine()
plt.show()

## 2. Partition plot helpers

In [ ]:
from src.utils_val import (
    best_by_metric,
    fmt_num,
    is_baseline_label,
    is_enet_label,
    is_xgb_label,
    label_family,
    make_partition_table,
    metric_name,
    plot_partition_bar,
    print_family_summary,
)

import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")


METRIC_NAMES = {
    "test_pearson_corr": "Pearson",
    "test_cosine_similarity": "cosine similarity",
    "test_mse": "MSE",
}

## 3. Final partition plots by validation dataset

In [ ]:
# Exclude exact k50 and k500 runs from this analysis.
# This keeps k5000.
skip_k_pattern = r"(?<![A-Za-z0-9])k(?:50|500)(?![A-Za-z0-9])"

skip_mask = (
    metrics_by_protein["run_label"]
    .astype(str)
    .str.contains(skip_k_pattern, case=False, regex=True, na=False)
)

metrics_partition = metrics_by_protein.loc[~skip_mask].copy()

print(
    f"Using {len(metrics_partition):,} rows after excluding "
    f"{skip_mask.sum():,} k50/k500 rows."
)

partition_tables = {}

for dataset_name in sorted(metrics_partition["dataset"].dropna().unique()):
    print("\n" + "=" * 110)
    print(f"DATASET: {dataset_name}")
    print("=" * 110)

    ds = metrics_partition[metrics_partition["dataset"] == dataset_name].copy()

    best_pearson = best_by_metric(ds, "test_pearson_corr", higher=True)
    best_cosine = best_by_metric(ds, "test_cosine_similarity", higher=True)
    best_mse = best_by_metric(ds, "test_mse", higher=False)

    mse_values = best_mse["test_mse"].dropna()

    mse_threshold = (
        float(mse_values.quantile(MSE_GOOD_PERCENTILE / 100.0))
        if len(mse_values)
        else np.nan
    )

    partition_tables[(dataset_name, "pearson")] = plot_partition_bar(
        best_pearson,
        metric="test_pearson_corr",
        threshold=PEARSON_GOOD_THRESHOLD,
        higher=True,
        title=f"{dataset_name}: best model by Pearson",
        min_proteins_to_show=MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT,
    )

    partition_tables[(dataset_name, "cosine")] = plot_partition_bar(
        best_cosine,
        metric="test_cosine_similarity",
        threshold=COSINE_GOOD_THRESHOLD,
        higher=True,
        title=f"{dataset_name}: best model by cosine similarity",
        min_proteins_to_show=MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT,
    )

    partition_tables[(dataset_name, "mse")] = plot_partition_bar(
        best_mse,
        metric="test_mse",
        threshold=mse_threshold,
        higher=False,
        title=f"{dataset_name}: best model by MSE",
        min_proteins_to_show=MIN_PROTEINS_TO_SHOW_AS_OWN_SEGMENT,
    )

## 4. Tissue-level summary of winning model families

In [ ]:
if metrics_by_tissue.empty:
    print("No tissue-level metrics were found. Run notebook 7 first.")
else:
    print("Tissue-level metric rows:", len(metrics_by_tissue))
    for dataset_name in sorted(metrics_by_tissue["dataset"].dropna().unique()):
        sub = metrics_by_tissue[metrics_by_tissue["dataset"] == dataset_name].copy()
        if sub.empty:
            continue
        best_tissue_pearson = (
            sub.dropna(subset=["test_pearson_corr"])
            .sort_values(["tissue", "test_pearson_corr"], ascending=[True, False])
            .groupby("tissue", as_index=False)
            .head(1)
        )
        print("\n", dataset_name, "best tissue-level Pearson winners")
        display(best_tissue_pearson[["tissue", "run_label", "test_pearson_corr", "n_samples", "n_proteins"]].sort_values("test_pearson_corr", ascending=False).head(20))

        fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * best_tissue_pearson["tissue"].nunique())))
        plot_data = best_tissue_pearson.sort_values("test_pearson_corr", ascending=True).tail(25)
        sns.barplot(data=plot_data, y="tissue", x="test_pearson_corr", hue="run_label", dodge=False, ax=ax)
        ax.set_title(f"{dataset_name}: best run per tissue by Pearson")
        ax.set_xlabel("Tissue-level Pearson across all protein/sample pairs")
        ax.set_ylabel("Tissue")
        ax.legend(loc="lower right", fontsize=8, frameon=True)
        plt.tight_layout()
        plt.show()


## 5. Optional exports

In [ ]:
# Optional: save partition tables.
# Uncomment if needed.
# out_dir = OUTPUT_DIR / "partition_tables"
# out_dir.mkdir(parents=True, exist_ok=True)
# for (dataset_name, metric_name), table in partition_tables.items():
#     table.to_csv(out_dir / f"partition_{dataset_name}_{metric_name}.csv", index=False)
# print("Saved partition tables to", out_dir)


## 6. ENet z-score k5000 vs scTranslator 10K predictions

This section compares the saved validation results for `enet_zscore k2000` against the `predictions_10K` scTranslator outputs stored in:

- `itraq/predictions_10K.npz` or `.npy`
- `labelfree/predictions_10K.npz` or `.npy`
- `tmt/predictions_10K.npz` or `.npy`

The comparison is done protein-by-protein on the same validation targets using:

- Pearson correlation
- Cosine similarity
- MSE

For ENet, predictions and targets are read from the long validation prediction file saved by notebook 7.
For scTranslator, predictions are loaded from the dataset-specific `predictions_10K` files.

In [ ]:
from src.utils_val import first_existing_column, is_enet_zbig_zscore_k5000_label, load_eval_predictions_long, normalize_dataset_name

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SCTRANSLATOR_VARIANT = "10K"

SCTRANSLATOR_DATASETS = {
    "itraq": "itraq",
    "labelfree": "labelfree",
    "tmt": "tmt",
}

TARGET_ENET_RUN_LABEL = "res_enet_zbig | zscore | fast | k5000"

ENET_MODEL_LABEL = "ENet zbig z-score k5000"
SCTRANSLATOR_MODEL_LABEL = "scTranslator 10K"










uv.configure_runtime_globals(OUTPUT_DIR=OUTPUT_DIR, PREDICTIONS_PARQUET=PREDICTIONS_PARQUET, PREDICTIONS_CSV=PREDICTIONS_CSV, TARGET_ENET_RUN_LABEL=TARGET_ENET_RUN_LABEL)

eval_predictions_long = load_eval_predictions_long()

DATASET_COL = first_existing_column(
    eval_predictions_long,
    ["dataset"],
)

RUN_COL = first_existing_column(
    eval_predictions_long,
    ["run_label", "model", "model_label"],
)

PROTEIN_COL = first_existing_column(
    eval_predictions_long,
    ["protein", "protein_id", "target", "gene"],
)

TRUE_COL = first_existing_column(
    eval_predictions_long,
    [
        "y_true",
        "true",
        "truth",
        "target_value",
        "observed",
        "observed_value",
        "value_true",
        "test_y",
        "y",
    ],
)

PRED_COL = first_existing_column(
    eval_predictions_long,
    [
        "y_pred",
        "pred",
        "prediction",
        "predicted",
        "predicted_value",
        "value_pred",
        "test_pred",
        "y_hat",
    ],
)

SAMPLE_COL = first_existing_column(
    eval_predictions_long,
    [
        "sample",
        "sample_id",
        "cell_line",
        "cell_line_id",
        "depmap_id",
        "DepMap_ID",
        "model_id",
        "sample_name",
    ],
    required=False,
)

print("Detected columns")
print("- dataset:", DATASET_COL)
print("- run label:", RUN_COL)
print("- protein:", PROTEIN_COL)
print("- true value:", TRUE_COL)
print("- predicted value:", PRED_COL)
print("- sample:", SAMPLE_COL if SAMPLE_COL is not None else "not found; using row order within protein")

uv.configure_runtime_globals(
    OUTPUT_DIR=OUTPUT_DIR,
    PREDICTIONS_PARQUET=PREDICTIONS_PARQUET,
    PREDICTIONS_CSV=PREDICTIONS_CSV,
    TARGET_ENET_RUN_LABEL=TARGET_ENET_RUN_LABEL,
)

In [ ]:
for dataset_key in ["itraq", "labelfree", "tmt"]:
    print("\n" + "=" * 90)
    print(dataset_key)

    tmp = eval_predictions_long.copy()
    tmp["_dataset_norm"] = tmp[DATASET_COL].map(normalize_dataset_name)

    ds = tmp[
        tmp["_dataset_norm"] == normalize_dataset_name(dataset_key)
    ].copy()

    print("Rows:", len(ds))

    print("Available run labels:")
    display(
        ds[[RUN_COL]]
        .drop_duplicates()
        .sort_values(RUN_COL)
        .reset_index(drop=True)
    )

    print("Matched ENet zbig z-score k5000 labels:")
    display(
        ds.loc[
            ds[RUN_COL].map(is_enet_zbig_zscore_k5000_label),
            [RUN_COL],
        ]
        .drop_duplicates()
        .sort_values(RUN_COL)
        .reset_index(drop=True)
    )

In [ ]:
from src.utils_val import (
    align_sctranslator_predictions,
    build_enet_matrices_for_dataset,
    clean_label_list,
    extract_labels_from_metadata,
    extract_prediction_array_from_dict,
    find_sctranslator_prediction_file,
    load_prediction_array,
    per_protein_metrics,
)

In [ ]:
from src.utils_val import build_enet_matrices_for_dataset, is_enet_zbig_zscore_k5000_label

TARGET_ENET_RUN_LABEL = "res_enet_zbig | zscore | fast | k5000"
ENET_MODEL_LABEL = "ENet zbig z-score k5000"
SCTRANSLATOR_MODEL_LABEL = "scTranslator 10K"

In [ ]:
comparison_metric_tables = []
head_to_head_tables = []

for dataset_key, dataset_folder in SCTRANSLATOR_DATASETS.items():
    print("\n" + "=" * 110)
    print(f"DATASET: {dataset_key}")
    print("=" * 110)

    enet_bundle = build_enet_matrices_for_dataset(
        eval_predictions_long,
        dataset_key,
    )

    print("Selected ENet run:", enet_bundle["selected_run"])
    print("ENet validation matrix:", enet_bundle["y_true"].shape)

    pred_path = find_sctranslator_prediction_file(
        dataset_folder,
        variant=SCTRANSLATOR_VARIANT,
    )

    print("scTranslator prediction file:", pred_path)

    sct_array, sct_metadata = load_prediction_array(pred_path)

    print("Raw scTranslator matrix:", sct_array.shape)

    if len(sct_metadata):
        print("scTranslator metadata keys:", list(sct_metadata.keys()))

    y_true, enet_pred, sct_pred, proteins = align_sctranslator_predictions(
        sct_array,
        sct_metadata,
        enet_bundle,
    )

    print("Aligned matrix shape:", y_true.shape)

    enet_metrics = per_protein_metrics(
        y_true,
        enet_pred,
        proteins,
    )

    enet_metrics["dataset"] = dataset_key
    enet_metrics["model"] = ENET_MODEL_LABEL

    sct_metrics = per_protein_metrics(
        y_true,
        sct_pred,
        proteins,
    )

    sct_metrics["dataset"] = dataset_key
    sct_metrics["model"] = SCTRANSLATOR_MODEL_LABEL

    comparison_metric_tables.extend([enet_metrics, sct_metrics])

    merged = enet_metrics.merge(
        sct_metrics,
        on=["dataset", "protein"],
        suffixes=("_enet", "_sctranslator"),
    )

    merged["delta_pearson_sctranslator_minus_enet"] = (
        merged["test_pearson_corr_sctranslator"]
        - merged["test_pearson_corr_enet"]
    )

    merged["delta_cosine_sctranslator_minus_enet"] = (
        merged["test_cosine_similarity_sctranslator"]
        - merged["test_cosine_similarity_enet"]
    )

    merged["delta_mse_sctranslator_minus_enet"] = (
        merged["test_mse_sctranslator"]
        - merged["test_mse_enet"]
    )

    merged["pearson_winner"] = np.select(
        [
            merged["test_pearson_corr_sctranslator"] > merged["test_pearson_corr_enet"],
            merged["test_pearson_corr_sctranslator"] < merged["test_pearson_corr_enet"],
        ],
        [
            SCTRANSLATOR_MODEL_LABEL,
            ENET_MODEL_LABEL,
        ],
        default="Tie",
    )

    merged["cosine_winner"] = np.select(
        [
            merged["test_cosine_similarity_sctranslator"] > merged["test_cosine_similarity_enet"],
            merged["test_cosine_similarity_sctranslator"] < merged["test_cosine_similarity_enet"],
        ],
        [
            SCTRANSLATOR_MODEL_LABEL,
            ENET_MODEL_LABEL,
        ],
        default="Tie",
    )

    merged["mse_winner"] = np.select(
        [
            merged["test_mse_sctranslator"] < merged["test_mse_enet"],
            merged["test_mse_sctranslator"] > merged["test_mse_enet"],
        ],
        [
            SCTRANSLATOR_MODEL_LABEL,
            ENET_MODEL_LABEL,
        ],
        default="Tie",
    )

    head_to_head_tables.append(merged)

comparison_metrics = pd.concat(comparison_metric_tables, ignore_index=True)
head_to_head_comparison = pd.concat(head_to_head_tables, ignore_index=True)

print("\nCombined per-protein comparison metrics:")
display(comparison_metrics.head())

print("\nHead-to-head comparison:")
display(head_to_head_comparison.head())

In [ ]:
comparison_summary = (
    comparison_metrics
    .groupby(["dataset", "model"])
    .agg(
        proteins=("protein", "nunique"),
        median_pearson=("test_pearson_corr", "median"),
        median_cosine=("test_cosine_similarity", "median"),
        median_mse=("test_mse", "median"),
        mean_pearson=("test_pearson_corr", "mean"),
        mean_cosine=("test_cosine_similarity", "mean"),
        mean_mse=("test_mse", "mean"),
    )
    .reset_index()
)

print("ENet zbig z-score k5000 vs scTranslator 10K summary")
display(
    comparison_summary.sort_values(
        ["dataset", "median_pearson"],
        ascending=[True, False],
    )
)

winner_summary = (
    head_to_head_comparison
    .groupby("dataset")
    .agg(
        proteins=("protein", "nunique"),

        enet_pearson_wins=(
            "pearson_winner",
            lambda x: int((x == ENET_MODEL_LABEL).sum()),
        ),
        sctranslator_pearson_wins=(
            "pearson_winner",
            lambda x: int((x == SCTRANSLATOR_MODEL_LABEL).sum()),
        ),
        pearson_ties=(
            "pearson_winner",
            lambda x: int((x == "Tie").sum()),
        ),

        enet_cosine_wins=(
            "cosine_winner",
            lambda x: int((x == ENET_MODEL_LABEL).sum()),
        ),
        sctranslator_cosine_wins=(
            "cosine_winner",
            lambda x: int((x == SCTRANSLATOR_MODEL_LABEL).sum()),
        ),
        cosine_ties=(
            "cosine_winner",
            lambda x: int((x == "Tie").sum()),
        ),

        enet_mse_wins=(
            "mse_winner",
            lambda x: int((x == ENET_MODEL_LABEL).sum()),
        ),
        sctranslator_mse_wins=(
            "mse_winner",
            lambda x: int((x == SCTRANSLATOR_MODEL_LABEL).sum()),
        ),
        mse_ties=(
            "mse_winner",
            lambda x: int((x == "Tie").sum()),
        ),

        median_delta_pearson=(
            "delta_pearson_sctranslator_minus_enet",
            "median",
        ),
        median_delta_cosine=(
            "delta_cosine_sctranslator_minus_enet",
            "median",
        ),
        median_delta_mse=(
            "delta_mse_sctranslator_minus_enet",
            "median",
        ),
    )
    .reset_index()
)

for col in [
    "enet_pearson_wins",
    "sctranslator_pearson_wins",
    "pearson_ties",
    "enet_cosine_wins",
    "sctranslator_cosine_wins",
    "cosine_ties",
    "enet_mse_wins",
    "sctranslator_mse_wins",
    "mse_ties",
]:
    winner_summary[col + "_percent"] = (
        100 * winner_summary[col] / winner_summary["proteins"]
    ).round(1)

print("Head-to-head winner summary")
display(winner_summary)

In [ ]:
plot_summary = comparison_summary.copy()

metric_specs = [
    ("median_pearson", "Median Pearson", False),
    ("median_cosine", "Median cosine similarity", False),
    ("median_mse", "Median MSE", True),
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4.5),
    constrained_layout=True,
)

for ax, (metric, title, use_log) in zip(axes, metric_specs):
    sns.barplot(
        data=plot_summary,
        x="dataset",
        y=metric,
        hue="model",
        ax=ax,
        errorbar=None,
        width=0.72,
    )

    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(title)

    if use_log:
        ax.set_yscale("log")

    ax.grid(axis="x", visible=False)

    if ax.get_legend() is not None:
        ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title="Model",
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
)

sns.despine()
plt.show()

In [ ]:
# Optional: save ENet z-score k2000 vs scTranslator 10K comparison outputs.
# Uncomment if needed.

# comparison_out_dir = OUTPUT_DIR / "enet_zscore_k2000_vs_sctranslator_10K"
# comparison_out_dir.mkdir(parents=True, exist_ok=True)

# comparison_metrics.to_csv(
#     comparison_out_dir / "metrics_by_protein_long.csv",
#     index=False,
# )

# head_to_head_comparison.to_csv(
#     comparison_out_dir / "head_to_head_by_protein.csv",
#     index=False,
# )

# comparison_summary.to_csv(
#     comparison_out_dir / "summary_by_dataset_model.csv",
#     index=False,
# )

# winner_summary.to_csv(
#     comparison_out_dir / "winner_summary_by_dataset.csv",
#     index=False,
# )

# print("Saved comparison outputs to:", comparison_out_dir)